We preprocess the three datasets we use for evaluation: neftel_ss2, ji_skin, and kim_lung.

# Neftel_ss2

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np


In [19]:
adata_full = sc.read_h5ad("./data/Neftel_ss2/nef_raw.h5ad")
display(adata_full.obs.describe())
print(adata_full.X)
print(adata_full.obs['cell_type'])

,genes_expressed,MESlike2,MESlike1,AClike,OPClike,NPClike1,NPClike2,G1S,G2M,tSNE1,...,targeted_rx_response,ICB_exposed,ICB_response,ET_exposed,ET_response,time_end_of_rx_to_sampling,post_sampling_rx_exposed,post_sampling_rx_response,PFS_DFS,OS
count,7930.000000,6576.000000,6576.000000,6576.000000,6576.000000,6576.000000,6576.000000,6576.000000,6576.000000,7930.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,5735.108953,0.024542,0.042822,0.046768,-0.029441,-0.029856,-0.008217,-0.012842,-0.005317,1.104803,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1422.089176,0.806760,1.116146,1.375818,0.875517,0.959581,0.983590,1.153580,0.893094,12.969348,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,3003.000000,-1.331215,-1.873096,-3.057608,-2.110158,-2.346176,-1.847342,-1.501211,-1.202059,-31.165455,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,4639.250000,-0.487077,-0.875707,-1.086154,-0.665988,-0.762426,-0.709006,-0.731377,-0.451206,-4.724205,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,5661.000000,-0.176491,-0.182638,-0.054250,-0.080703,-0.059586,-0.209462,-0.467789,-0.282051,1.107272,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,6752.000000,0.245840,0.721042,1.169127,0.500535,0.733500,0.355404,0.161912,-0.024298,11.253733,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,11890.000000,4.109862,4.749972,4.180027,3.571748,2.898836,3.888995,4.233588,4.993003,24.065902,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


  (0, 5)	0.46000000834465027
  (0, 20)	1.0299999713897705
  (0, 25)	6.300000190734863
  (0, 27)	1.440000057220459
  (0, 31)	0.9800000190734863
  (0, 33)	310.8999938964844
  (0, 36)	52.18000030517578
  (0, 45)	0.18000000715255737
  (0, 49)	3.0999999046325684
  (0, 67)	53.529998779296875
  (0, 73)	58.709999084472656
  (0, 77)	124.27999877929688
  (0, 80)	0.9700000286102295
  (0, 89)	416.5199890136719
  (0, 96)	3.119999885559082
  (0, 106)	0.6399999856948853
  (0, 110)	1.5099999904632568
  (0, 124)	0.1599999964237213
  (0, 133)	1.440000057220459
  (0, 137)	1.190000057220459
  (0, 143)	1.4299999475479126
  (0, 145)	6.210000038146973
  (0, 148)	112.7699966430664
  (0, 171)	191.32000732421875
  (0, 177)	923.6900024414062
  :	:
  (7929, 23612)	1.6399999856948853
  (7929, 23613)	186.5
  (7929, 23614)	5.429999828338623
  (7929, 23616)	1.8899999856948853
  (7929, 23617)	1.0099999904632568
  (7929, 23618)	46.599998474121094
  (7929, 23622)	122.61000061035156
  (7929, 23628)	1.149999976158142
  (7

These are not log1p normalised, so we do that first. Also, we select the Malignant cells. 

In [20]:
adata_log1p = sc.pp.log1p(adata_full, copy=True)
adata_log1p = adata_log1p[adata_log1p.obs['cell_type'] == 'Malignant']

Next, we select for cells that are not currently replicating, as those are very different in distribution from non-replicating cells. But then we're left with only 1 celltype. As a biological signal to use for clustering, we use the "MESlike2", "MESlike1", "AClike" etc columns.

In [21]:
# List of S-phase genes
s_genes = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UNG",
    "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1", "HELLS",
    "RFC2", "RPA2", "NASP", "RAD51AP1", "GMNN", "WDR76", "SLBP",
    "CCNE2", "UBR7", "POLD3", "MSH2", "ATAD2", "RAD51", "RRM2",
    "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2",
    "USP1", "CLSPN", "POLA1", "CHAF1B", "BRIP1", "E2F8"
]


# List of G2/M-phase genes
g2m_genes = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A",
    "NDC80", "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF",
    "TACC3", "FAM64A", "SMC4", "CCNB2", "CKAP2L", "CKAP2", "AURKB",
    "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1", "KIF20B", "HJURP",
    "CDCA3", "HN1", "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1",
    "NCAPD2", "DLGAP5", "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR",
    "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5", "CENPE", "CTCF",
    "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA"
]

sc.tl.score_genes_cell_cycle(adata_log1p, s_genes=s_genes, g2m_genes=g2m_genes)
adata_log1p = adata_log1p[
    adata_log1p.obs['phase'] == 'G1'  # Keep only G1 phase cells
]
adata_log1p = adata_log1p[adata_log1p.obs['MESlike1'].notna()]
adata_log1p.obs['MESlike'] = adata_log1p.obs[['MESlike1', 'MESlike2']].max(axis=1)
adata_log1p.obs['NPClike'] = adata_log1p.obs[['NPClike1', 'NPClike2']].max(axis=1)

subtype_columns = ['NPClike', 'MESlike', 'AClike', 'OPClike']

# Find the name of the column with the max value for each row and assign it to 'subtype'
adata_log1p.obs['subtype'] = adata_log1p.obs[subtype_columns].idxmax(axis=1)

/local/home/rvander/miniconda3/envs/cancerfoundation2/lib/python3.9/site-packages/scanpy/tools/_score_genes.py:169: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
/tmp/ipykernel_524639/932433671.py:29: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_log1p.obs['MESlike'] = adata_log1p.obs[['MESlike1', 'MESlike2']].max(axis=1)


That's neftel_ss2 done!

In [22]:
adata_log1p.write_h5ad("./data/Neftel_ss2/neftel_ss2.h5ad")

# Kim_lung

In [38]:
adata_full = sc.read_h5ad("./data/Kim_lung/kim_lung.h5ad")
display(adata_full.obs.describe())
print(adata_full.obs.keys())
print(adata_full.obs['celltype'])
print(adata_full.X)

,sample,celltype,phase
count,5876,5876,5876
unique,10,3,1
top,P0034,tS2,G1
freq,2291,2938,5876


Index(['sample', 'celltype', 'phase'], dtype='object')
cell_name
AAACCTGAGACTGTAA_LUNG_T34    tS2
AAACCTGAGATGTCGG_LUNG_T31    tS1
AAACCTGAGCTAAACA_LUNG_T34    tS1
AAACCTGAGCTACCTA_LUNG_T34    tS2
AAACCTGCAACTGCGC_LUNG_T31    tS2
                            ... 
TTTGTCAGTTTGTGTG_LUNG_T30    tS1
TTTGTCATCAGGCAAG_LUNG_T34    tS1
TTTGTCATCCAAAGTC_LUNG_T30    tS1
TTTGTCATCTTGCCGT_LUNG_T30    tS1
TTTGTCATCTTTAGGG_LUNG_T31    tS2
Name: celltype, Length: 5876, dtype: category
Categories (3, object): ['tS1', 'tS2', 'tS3']
  (0, 11)	0.26061782240867615
  (0, 12)	0.26061782240867615
  (0, 14)	0.6382657885551453
  (0, 18)	0.26061782240867615
  (0, 21)	0.4671640992164612
  (0, 31)	0.26061782240867615
  (0, 36)	0.26061782240867615
  (0, 38)	0.26061782240867615
  (0, 48)	0.26061782240867615
  (0, 50)	0.26061782240867615
  (0, 53)	1.6985423564910889
  (0, 61)	0.6382657885551453
  (0, 62)	0.26061782240867615
  (0, 63)	0.26061782240867615
  (0, 64)	0.7843244671821594
  (0, 66)	0.4671640992164612
  (0, 

It's already log1p normalised, we only have G1 phase cells, all cells are cancer cells, and there are cell_types we can use to cluster. So this dataset is already good!

# Ji_skin

In [43]:
adata_full = sc.read_h5ad("./data/Ji_skin/ji_skin_raw.h5ad")
# display(adata_full.var.describe())
display(adata_full.obs.describe())
print(adata_full.obs.keys())
print(adata_full.obs['celltype'])
print(adata_full.X)
print(adata_full.obs['subclones'])
# adata_full = adata_full[adata_full.obs["phase"] == "G1"]
# adata_full.write_h5ad("./data/Ji_skin/ji_skin_processed.h5ad")

,G2M_score,S_score
count,10529.000000,10529.000000
mean,-0.109389,-0.253431
std,0.916446,0.642363
min,-1.271366,-1.347414
25%,-0.676296,-0.707279
50%,-0.505890,-0.462994
75%,0.042466,0.018469
max,3.658220,2.324264


Index(['sample', 'celltype', 'subclones', 'phase', 'G2M_score', 'S_score'], dtype='object')
cell_name
P1-Tumor-AAACCTGAGTCAAGCG      diff
P1-Tumor-AAACCTGGTAGGAGTC     basal
P1-Tumor-AAACGGGAGATGTAAC     basal
P1-Tumor-AAACGGGAGCTGCAAG     basal
P1-Tumor-AAAGATGAGAAGGTTT     basal
                              ...  
P10-Tumor-TTTGCGCTCTTGTACT     diff
P10-Tumor-TTTGGTTGTTGGGACA    basal
P10-Tumor-TTTGGTTTCCCAACGG     diff
P10-Tumor-TTTGTCAAGCATGGCA      TSK
P10-Tumor-TTTGTCAGTGCCTGTG     diff
Name: celltype, Length: 10529, dtype: category
Categories (3, object): ['TSK', 'basal', 'diff']
  (0, 8)	1.7856011390686035
  (0, 10)	1.7856011390686035
  (0, 12)	3.431687355041504
  (0, 16)	3.9713804721832275
  (0, 17)	1.7856011390686035
  (0, 18)	1.7856011390686035
  (0, 23)	1.7856011390686035
  (0, 25)	3.431687355041504
  (0, 29)	2.5595510005950928
  (0, 32)	1.7856011390686035
  (0, 33)	1.7856011390686035
  (0, 34)	1.7856011390686035
  (0, 37)	4.924643516540527
  (0, 40)	3.7266247272491455
  (0

The counts are already normalised. We need to select for the phase, though. I think (?) all these cells are cancerous.

In [ ]:
adata_full = adata_full[adata_full.obs["phase"] == "G1"]
adata_full.write_h5ad("./data/Ji_skin/ji_skin.h5ad")